In [5]:
def count_vector(events: str) -> tuple:
    return tuple(sorted(Counter(events.split()).items()))

sequences["count_vector"] = sequences["events"].apply(count_vector)
for label in ("Normal", "Anomaly"):
    subset = sequences[sequences["Label"] == label]
    print(f"{label:<8} blocks {len(subset):>8,}   distinct event-count vectors {subset['count_vector'].nunique():>7,}   "
          f"distinct ordered sequences {subset['events'].nunique():>7,}")

Normal   blocks  558,223   distinct event-count vectors     212   distinct ordered sequences  14,259
Anomaly  blocks   16,838   distinct event-count vectors     394   distinct ordered sequences   4,126


In [4]:
present = (sequences[["BlockId", "Label"]]
           .assign(template_id=sequences["events"].str.split().apply(lambda ids: sorted(set(ids))))
           .explode("template_id"))
present["template_id"] = present["template_id"].astype(int)

counts = present.groupby(["template_id", "Label"]).size().unstack(fill_value=0)
n_blocks = sequences["Label"].value_counts()
share = pd.DataFrame({
    "in_normal_%": (100 * counts.get("Normal", 0) / n_blocks["Normal"]).round(2),
    "in_anomaly_%": (100 * counts.get("Anomaly", 0) / n_blocks["Anomaly"]).round(2),
}).join(templates.set_index("template_id")["template"])

share["difference"] = (share["in_anomaly_%"] - share["in_normal_%"]).round(2)
print("Templates never seen in any normal block:",
      share.index[share["in_normal_%"] == 0].tolist())
share.sort_values("difference", ascending=False)

Templates never seen in any normal block: [8, 9, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]


,in_normal_%,in_anomaly_%,template,difference
template_id,,,,
35,0.04,30.44,Unexpected error trying to delete block <BLK>. BlockInfo not found in volumeMap.,30.40
7,0.32,20.58,<IP>:Transmitted block <BLK> to /<IP>,20.26
6,0.32,20.58,Received block <BLK> src: /<IP> dest: /<IP> of size <NUM>,20.26
11,0.32,20.33,BLOCK* ask <IP> to replicate <BLK> to datanode(s) <IP>,20.01
12,0.32,20.33,<IP> Starting thread to transfer block <BLK> to <IP>,20.01
16,0.00,19.16,writeBlock <BLK> received exception java.io.IOException: Could not read from stream,19.16
36,0.00,7.41,BLOCK* NameSystem.addStoredBlock: addStoredBlock request received for <BLK> on <IP> size <NUM> But it does not belong to any f...,7.41
21,0.00,5.77,BLOCK* NameSystem.addStoredBlock: Redundant addStoredBlock request received for <BLK> on <IP> size <NUM>,5.77
18,0.00,2.90,Receiving empty packet for block <BLK>,2.90


In [3]:
reference = pd.read_csv(REF)
print("Reference columns:", reference.columns.tolist())
ref_column = next(c for c in reference.columns if "template" in c.lower())

def normalise(text: str) -> str:
    """Make templates comparable: every wildcard style (<NUM>, <*>, [*], *) becomes '*'."""
    text = re.sub(r"<[^>]*>|\[\*\]|\*", " * ", str(text))
    return " ".join(text.lower().split())

ours = [normalise(t) for t in templates["template"]]
rows = []
for ref_template in reference[ref_column]:
    target = normalise(ref_template)
    best = max(ours, key=lambda o: SequenceMatcher(None, target, o).ratio())
    rows.append((ref_template, best, round(SequenceMatcher(None, target, best).ratio(), 2)))

comparison = pd.DataFrame(rows, columns=["reference", "closest_ours", "similarity"]).sort_values("similarity")
print(f"Reference templates: {len(reference)}   Ours: {len(templates)}")
print(f"Reference templates with a close match (similarity >= 0.9): {(comparison['similarity'] >= 0.9).sum()}")
comparison

Reference columns: ['EventId', 'EventTemplate']
Reference templates: 29   Ours: 53
Reference templates with a close match (similarity >= 0.9): 22


,reference,closest_ours,similarity
9,[*]PacketResponder[*]Exception[*],packetresponder * * exception java.io.eofexception,0.69
16,[*]:Failed to transfer[*]to[*]got[*],"* starting thread to transfer block * to * , *",0.70
6,[*]writeBlock[*]received exception[*],writeblock * received exception java.io.eofexception,0.74
13,[*]Exception in receiveBlock for block[*],exception in receiveblock for block * java.io.eofexception,0.76
24,[*]BLOCK* ask[*]to replicate[*]to[*],block * ask * to replicate * to datanode(s) *,0.82
20,[*]Deleting block[*]file[*],deleting block * file * / *,0.88
21,[*]BLOCK* NameSystem[*]allocateBlock:[*],block * namesystem.allocateblock: * *,0.89
4,[*]Receiving block[*]src:[*]dest:[*],receiving block * src: / * dest: / *,0.91
5,[*]Received block[*]src:[*]dest:[*]of size[*],received block * src: / * dest: / * of size *,0.93
18,[*]Reopen Block[*],reopen block *,0.93


In [2]:
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.max_rows", 100)
templates.sort_values("count", ascending=False).reset_index(drop=True)

,template_id,template,count
0,1,Receiving block <BLK> src: /<IP> dest: /<IP>,1723232
1,5,BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is added to <BLK> size <NUM>,1719741
2,3,PacketResponder <NUM> for block <BLK> terminating,1706679
3,4,Received block <BLK> of size <NUM> from /<IP>,1706514
4,17,Deleting block <BLK> file <PATH>/<BLK>,1402047
5,34,BLOCK* NameSystem.delete: <BLK> is added to invalidSet of <IP>,1396174
6,2,BLOCK* NameSystem.allocateBlock: <PATH> <BLK>,575061
7,10,<IP> Served block <BLK> to /<IP>,428726
8,37,<IP>:Got exception while serving <BLK> to /<IP>:,356207
9,13,Verification succeeded for <BLK>,120036


In [1]:
import json
import re
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root not found")
    ROOT = ROOT.parent
OUT = ROOT / "data" / "processed"
REF = ROOT / "data" / "raw" / "HDFS_v1" / "preprocessed" / "HDFS.log_templates.csv"

templates = pd.read_csv(OUT / "hdfs_templates.csv")
sequences = pd.read_csv(OUT / "hdfs_sequences.csv")
info = json.loads((OUT / "hdfs_parse_info.json").read_text())

print(json.dumps(info, indent=2))
print(f"Total events in sequences: {sequences['length'].sum():,}")
print(f"Blocks: {len(sequences):,}   Labels: {sequences['Label'].value_counts().to_dict()}")

{
  "input": "data\\raw\\HDFS_v1\\HDFS.log",
  "lines_read": 11175629,
  "lines_unparsed": 0,
  "lines_without_block": 0,
  "blocks": 575061,
  "templates": 53,
  "drain_sim_threshold": 0.9,
  "drain_depth": 4,
  "masks": [
    "BLK",
    "IP",
    "UUID",
    "HEX",
    "PATH",
    "NUM"
  ],
  "seconds": 127.1
}
Total events in sequences: 11,175,629
Blocks: 575,061   Labels: {'Normal': 558223, 'Anomaly': 16838}


## Findings
- Full parse: 11,175,629 lines in 127 s (~88k lines/s), 0 unparsed, 53 templates (stable after ~9 M lines).
- Sanity check: allocateBlock occurs 575,061 times = number of blocks (each block allocated once).
- Reference comparison: 22 of 29 reference templates matched closely (≥0.9). The rest are coarser in the
  reference (e.g. one "writeBlock received exception [*]" vs ours split per exception type). All reference
  templates are covered; ours are finer-grained (53 vs 29).
- Templates 3, 4, 5 (write completed) occur in 100% of normal blocks but only ~63% of anomalous blocks:
  ~37% of anomalies are writes that never completed (the short sessions found on Day 12).
- 41 of 53 templates never occur in normal blocks; strongest: #16 "Could not read from stream" (19.2% of
  anomalies), #36 (7.4%), #21 (5.8%), #18 "empty packet" (2.9%). #26 "Interrupted." occurs only in
  anomalies, confirming the Drain threshold choice (0.9).
- Normal behaviour is highly repetitive: 558,223 normal blocks have only 212 distinct event-count vectors
  (14,259 distinct ordered sequences). Anomalies: 394 vectors, 4,126 sequences.
- Caution: "never seen in normal" here uses all labels. In experiments, seen/unseen must be computed from
  the training split only, to avoid label leakage.

# 02 — HDFS templates and event sequences

Input: data/processed/hdfs_templates.csv and hdfs_sequences.csv (from experiments/parse_hdfs.py).
- Templates found by Drain, checked against the dataset authors' template list
- Which templates occur in normal vs anomalous blocks
- How repetitive normal behaviour is